# Customer Churn Analysis and Prediction Using Machine Learning

## CODTECH Internship Task 3

This notebook demonstrates a machine learning pipeline to predict customer churn using a Random Forest Classifier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

sns.set_theme(style="whitegrid")

### 1. Data Loading

In [ ]:
df = pd.read_csv('data/customer_churn.csv')
display(df.head())

### 2. Data Cleaning & Missing Value Handling

In [ ]:
if 'TotalCharges' in df.columns and df['TotalCharges'].dtype == 'object':
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

if 'CustomerID' in df.columns:
    df = df.drop(columns=['CustomerID'])

### 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Churn Distribution Analysis')
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True, palette='Set1')
plt.title('Monthly Charges Distribution by Churn')
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='Tenure', data=df, palette='Set3')
plt.title('Tenure Analysis by Churn')
plt.show()

### 4. Feature Engineering & Label Encoding

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le
display(df.head())

### 5. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

### 6. Train-Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

### 7. Model Training (Random Forest Classifier)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

### 8. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### 9. Feature Importance

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importance")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=45)
plt.tight_layout()
plt.show()

### 10. Prediction System

In [ ]:
sample = X_test.iloc[[0]]
prediction = model.predict(sample)
pred_label = le_dict['Churn'].inverse_transform(prediction)
print(f"Sample predicted as: {pred_label[0]}")